In [1]:
#import useful python libraries
import numpy as np
import pandas as pd
import light_curve as lc
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix


In [2]:
#loading dataset
df = pd.read_parquet("masked_objects.parquet")

In [3]:
df.keys()

Index(['objectId', 'n_det', 'ra', 'dec', 'finkclass', 'filters_used'], dtype='object')

In [4]:
df.head(20)

,objectId,n_det,ra,dec,finkclass,filters_used
0,ZTF17aaaaaao,1,63.989645,45.430759,Unknown,[r]
1,ZTF17aaaaabl,1,69.926065,51.428767,Unknown,[r]
2,ZTF17aaaaaes,1,77.541800,47.395606,Unknown,[r]
3,ZTF17aaaaafy,9,67.117201,45.250294,Unknown,[r]
4,ZTF17aaaaafz,23,67.309158,45.409661,Unknown,"[g, r]"
5,ZTF17aaaaajf,1,47.593230,46.883066,Unknown,[g]
6,ZTF17aaaaakt,2,320.412997,49.523902,Unknown,"[g, r]"
7,ZTF17aaaaalb,1,326.824445,47.487003,Unknown,[r]
8,ZTF17aaaaamo,1,54.370615,48.471151,Unknown,[g]
9,ZTF17aaaaapa,1,59.530935,45.576800,Unknown,[g]


In [5]:
df.shape


(989256, 6)

In [6]:
#check the elements
df['finkclass'].unique()

['Unknown', 'CataclyV*', 'CataclyV*_Candidate']
Categories (163, object): ['**', 'AGB*', 'AGB*_Candidate', 'AGN', ..., 'post-AGB*', 'post-AGB*_Candidate', 'radioBurst', 'smmRad']

In [7]:
#count of element distribution
print(df['finkclass'].value_counts())
print("Total objects:", len(df))

finkclass
Unknown                 987841
CataclyV*                 1183
CataclyV*_Candidate        232
**                           0
RedSG                        0
                         ...  
HighVel*                     0
HorBranch*                   0
HorBranch*_Candidate         0
HotSubdwarf                  0
smmRad                       0
Name: count, Length: 163, dtype: int64
Total objects: 989256


In [8]:
#picking subset
df['finkclass'] = df['finkclass'].str.strip()
unknown_sample = df[df['finkclass'] == 'Unknown'] \
                    .sample(n=1200, random_state=42)

cataclyv_sample = df[df['finkclass'] == 'CataclyV*'] \
                     .sample(n=800, random_state=42)

candidate_sample = df[df['finkclass'] == 'CataclyV*_Candidate']
# No sampling needed since total = 232

In [9]:
#saving
final_df = pd.concat(
    [unknown_sample, cataclyv_sample, candidate_sample]
)

In [10]:
#double check
print(final_df['finkclass'].value_counts())
print("Total:", len(final_df))

finkclass
Unknown                1200
CataclyV*               800
CataclyV*_Candidate     232
Name: count, dtype: int64
Total: 2232


In [11]:
final_df.to_parquet("subset_sample.parquet", index=False)

In [12]:
final_df

,objectId,n_det,ra,dec,finkclass,filters_used
736593,ZTF21aadodix,1,106.533739,-5.310724,Unknown,[g]
910456,ZTF25acisuht,1,79.439400,-25.633381,Unknown,[r]
262099,ZTF18acnkfxq,1,35.983899,53.904306,Unknown,[g]
674563,ZTF20abjgjha,9,46.750326,53.124241,Unknown,"[g, r]"
801337,ZTF22aalkfpw,1,264.614866,3.804705,Unknown,[r]
...,...,...,...,...,...,...
812505,ZTF22abjpciz,1,72.180288,-12.051509,CataclyV*_Candidate,[r]
817792,ZTF22abshymy,3,67.064925,-1.781685,CataclyV*_Candidate,[r]
838536,ZTF23aailiep,6,243.312951,3.207429,CataclyV*_Candidate,"[g, r]"
870437,ZTF24aalgmbp,1,198.727616,-5.587236,CataclyV*_Candidate,[g]


In [13]:
#working with the r-band data only
df_lc=pd.read_parquet('fink_grouped_rband.parquet')

In [14]:
df_lc

,objectId,i:jd,i:magpsf,i:sigmapsf
0,ZTF17aaaadkj,"[2461075.7216782, 2461073.6943981, 2461071.736...","[18.114567, 18.40981, 18.30638, 18.030252, 18....","[0.05341228, 0.073376365, 0.10485197, 0.078975..."
1,ZTF17aaaakhm,"[2461049.7235764, 2460858.9818519, 2460733.662...","[19.321436, 19.681917, 19.899174, 19.558851, 1...","[0.17050926, 0.20623723, 0.17487903, 0.1726738..."
2,ZTF17aaaarmr,"[2461075.7221528, 2461071.6805903, 2461059.675...","[17.412136, 18.256882, 17.896376, 17.472588, 1...","[0.05556723, 0.102200404, 0.09755261, 0.041627..."
3,ZTF17aaaazob,"[2461055.618669, 2461023.6827199, 2461022.6201...","[19.856903, 19.137974, 18.623093, 18.90311, 18...","[0.19144532, 0.16273436, 0.12504944, 0.1056603..."
4,ZTF17aaaazwi,"[2461094.6225463, 2461061.6834722, 2461059.680...","[18.46709, 18.172516, 17.896368, 17.5138, 17.9...","[0.06487709, 0.054149523, 0.052875698, 0.04135..."
...,...,...,...,...
1966,ZTF26aagzjvi,[2461093.6264583],[19.051485],[0.19713093]
1967,ZTF26aagzkvu,[2461093.6264583],[19.179848],[0.12858544]
1968,ZTF26aahfqox,[2461094.7343981],[19.257664],[0.14651765]
1969,ZTF26aahkymp,"[2461101.055787, 2461101.0553125, 2461100.0589...","[16.384829, 16.366926, 16.358637, 16.260105, 1...","[0.029489188, 0.02961291, 0.042901866, 0.03462..."


In [15]:
#runthrough the environment
import sys
print(sys.executable)

D:\Mcvs_clermont\current working directory\cluster_groups\fink_env\Scripts\python.exe


In [16]:

# Load files
lightcurves = pd.read_parquet("fink_grouped_rband.parquet")
labels_df   = pd.read_parquet("subset_sample.parquet")

# Keep only needed columns
labels_df = labels_df[["objectId", "finkclass"]].drop_duplicates()

# Merge by objectId
merged_df = lightcurves.merge(
    labels_df,
    on="objectId",
    how="left"
)

# check how many matched
print("Total lightcurves:", len(lightcurves))
print("Matched classes:", merged_df["finkclass"].notna().sum())

# Save
merged_df.to_parquet("fink_grouped_rband_with_class.parquet", index=False)

Total lightcurves: 1971
Matched classes: 1971


In [17]:
df_new = pd.read_parquet("fink_grouped_rband_with_class.parquet")

In [18]:
df_new

,objectId,i:jd,i:magpsf,i:sigmapsf,finkclass
0,ZTF17aaaadkj,"[2461075.7216782, 2461073.6943981, 2461071.736...","[18.114567, 18.40981, 18.30638, 18.030252, 18....","[0.05341228, 0.073376365, 0.10485197, 0.078975...",CataclyV*
1,ZTF17aaaakhm,"[2461049.7235764, 2460858.9818519, 2460733.662...","[19.321436, 19.681917, 19.899174, 19.558851, 1...","[0.17050926, 0.20623723, 0.17487903, 0.1726738...",CataclyV*_Candidate
2,ZTF17aaaarmr,"[2461075.7221528, 2461071.6805903, 2461059.675...","[17.412136, 18.256882, 17.896376, 17.472588, 1...","[0.05556723, 0.102200404, 0.09755261, 0.041627...",CataclyV*
3,ZTF17aaaazob,"[2461055.618669, 2461023.6827199, 2461022.6201...","[19.856903, 19.137974, 18.623093, 18.90311, 18...","[0.19144532, 0.16273436, 0.12504944, 0.1056603...",CataclyV*
4,ZTF17aaaazwi,"[2461094.6225463, 2461061.6834722, 2461059.680...","[18.46709, 18.172516, 17.896368, 17.5138, 17.9...","[0.06487709, 0.054149523, 0.052875698, 0.04135...",CataclyV*_Candidate
...,...,...,...,...,...
1966,ZTF26aagzjvi,[2461093.6264583],[19.051485],[0.19713093],Unknown
1967,ZTF26aagzkvu,[2461093.6264583],[19.179848],[0.12858544],Unknown
1968,ZTF26aahfqox,[2461094.7343981],[19.257664],[0.14651765],Unknown
1969,ZTF26aahkymp,"[2461101.055787, 2461101.0553125, 2461100.0589...","[16.384829, 16.366926, 16.358637, 16.260105, 1...","[0.029489188, 0.02961291, 0.042901866, 0.03462...",CataclyV*


In [19]:
# Load grouped lightcurve file (WITH finkclass)
df_new = pd.read_parquet("fink_grouped_rband_with_class.parquet")

# Define extractor
extractor = lc.Extractor(
    lc.Mean(),
    lc.WeightedMean(),
    lc.StandardDeviation(),
    lc.Median(),
    lc.Amplitude(),
    lc.BeyondNStd(nstd=1),
    lc.Cusum(),
    lc.InterPercentileRange(0.10),
    lc.Kurtosis(),
    lc.LinearTrend(),
    lc.LinearFit(),
    lc.MagnitudePercentageRatio(0.4, .05),
    lc.MagnitudePercentageRatio(0.2, 0.1),
    lc.MaximumSlope(),
    lc.MedianAbsoluteDeviation(),
    lc.MedianBufferRangePercentage(0.10),
    lc.PercentAmplitude(),
    lc.MeanVariance(),
    lc.AndersonDarlingNormal(),
    lc.ReducedChi2(),
    lc.Skew(),
    lc.StetsonK()
)

# Feature extraction
results = []

for _, row in df_new.iterrows():  
    try:
        t = np.array(row["i:jd"])
        m = np.array(row["i:magpsf"])
        err = np.array(row["i:sigmapsf"])

        feats = extractor(t, m, err, sorted=True, check=False)

        # Append finkclass too
        results.append(
            [row["objectId"], row["finkclass"]] + list(feats)
        )

    except Exception as e:
        print(f"Skipping {row['objectId']} due to error: {e}")

# Create feature dataframe
features_df = pd.DataFrame(
    results,
    columns=["objectId", "finkclass"] + extractor.names
)

# Save
features_df.to_parquet("new_feature_space.parquet", index=False)

print("✅ Feature extraction complete!")
print("Saved to: new_feature_space.parquet")
print(features_df.shape)

Skipping ZTF17aaaftka due to error: time-series' length 1 is smaller than the minimum required length 2
Skipping ZTF17aaaftll due to error: time-series' length 1 is smaller than the minimum required length 2
Skipping ZTF17aaaxjkv due to error: time-series' length 1 is smaller than the minimum required length 2
Skipping ZTF17aaaypqv due to error: time-series' length 1 is smaller than the minimum required length 2
Skipping ZTF17aaazdaf due to error: time-series' length 2 is smaller than the minimum required length 4
Skipping ZTF17aacdeds due to error: time-series' length 2 is smaller than the minimum required length 4
Skipping ZTF17aacrito due to error: time-series' length 3 is smaller than the minimum required length 4
Skipping ZTF17aacrmkb due to error: time-series' length 1 is smaller than the minimum required length 2
Skipping ZTF17aacxvol due to error: time-series' length 1 is smaller than the minimum required length 2
Skipping ZTF17aacxwyv due to error: time-series' length 1 is sma

In [20]:
df_fea=pd.read_parquet("new_feature_space.parquet")

In [21]:
df_fea

,objectId,finkclass,mean,weighted_mean,standard_deviation,median,amplitude,beyond_1_std,cusum,inter_percentile_range_10,...,magnitude_percentage_ratio_20_10,maximum_slope,median_absolute_deviation,median_buffer_range_percentage_10,percent_amplitude,mean_variance,anderson_darling_normal,chi2,skew,stetson_K
0,ZTF17aaaadkj,CataclyV*,18.484376,18.013530,0.593808,18.495530,2.033953,0.149254,0.147635,0.989651,...,0.668111,778.715272,0.276399,0.368159,2.897469,0.032125,8.106756,167.142706,-2.096349,0.622763
1,ZTF17aaaakhm,CataclyV*_Candidate,19.523426,19.299517,0.428201,19.489155,1.097642,0.172414,0.155945,0.670212,...,0.733629,17.502317,0.141145,0.379310,1.275418,0.021933,1.547621,12.427528,-1.086096,0.654523
2,ZTF17aaaarmr,CataclyV*,17.558560,17.344502,0.440666,17.479678,1.323276,0.252101,0.203856,1.000527,...,0.630017,275.295240,0.220042,0.344538,1.411201,0.025097,3.421733,61.721443,0.441170,0.707557
3,ZTF17aaaazob,CataclyV*,19.060615,18.976308,0.372482,19.041963,1.092818,0.283784,0.158434,0.948873,...,0.635982,1673.060048,0.268435,0.243243,1.194768,0.019542,0.331244,7.886530,0.024982,0.788128
4,ZTF17aaaazwi,CataclyV*_Candidate,18.492340,18.099113,0.738607,18.304562,1.348280,0.363636,0.265114,1.893409,...,0.809566,40.722106,0.376825,0.121212,1.905799,0.039941,1.706506,40.696826,0.796089,0.829948
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1364,ZTF25absjzpb,CataclyV*,17.903463,17.901019,0.029894,17.896685,0.032739,0.250000,0.384976,0.065478,...,0.821827,0.032475,0.016239,0.000000,0.046295,0.001670,0.121673,0.139859,0.908388,0.903283
1365,ZTF25abtlopi,CataclyV*,18.446089,18.192322,0.493012,18.423594,0.504202,0.250000,0.267081,1.008404,...,0.896764,0.790686,0.394954,0.000000,0.549192,0.026727,0.139866,19.213541,0.108774,0.904742
1366,ZTF25acguadr,Unknown,17.485748,17.480386,0.125774,17.490637,0.152608,0.333333,0.435065,0.296116,...,0.815621,57.192278,0.107110,0.000000,0.155247,0.007193,0.300690,2.679935,-0.005940,0.933378
1367,ZTF26aaajqnu,CataclyV*,17.808031,17.804974,0.148051,17.782119,0.202300,0.375000,0.349516,0.373440,...,0.771488,96.766732,0.099034,0.000000,0.228183,0.008314,0.333754,3.314708,0.209490,0.857556
